
# 🧱 Databricks Course — Day 4 Notes
> **Focus:** Apache Spark ETL Overview, GizmoBox Project Setup, Querying Files & Views

---

## 1. ⚡ ETL with Apache Spark — What We'll Build

ETL = **Extract → Transform → Load**

The full scope of what Spark ETL covers in this course:

| Stage | What happens |
|---|---|
| **Extract** | Query files/folders, access external tables (Azure SQL, MySQL via JDBC) |
| **Validate** | Clean data, fix issues, handle nulls/duplicates |
| **Transform** | Simple & complex transformations, UDFs, JSON structures |
| **Load** | Create Delta tables, build business-level aggregates |

### Data Sources We'll Work With:

| Source | Format | How |
|---|---|---|
| Customers | JSON | Direct file query |
| Orders | JSON | Direct file query |
| Addresses | CSV/TSV | Direct file query |
| Memberships | Images (PNG) | Binary file format |
| Payments | CSV | Direct file query |
| Refunds | Azure SQL Database | JDBC connection |

### Techniques Covered:
- Direct file queries (SQL & PySpark)
- External tables & Views
- Simple & complex nested JSON structures
- Data validation & fixing issues
- Simple transformations
- User Defined Functions (UDFs)
- Creating Delta tables
- Both Spark SQL and PySpark approaches

---

## 2. 🛍️ GizmoBox Project — Overview

> An e-commerce company selling electronics direct to consumer. They want to build a **Data Lakehouse using Apache Spark** on Azure Databricks.

### Data Sources:

```
Operational Data (internal systems)        External Data
─────────────────────────────────          ──────────────────────────
Customers    → JSON                        Payments  → CSV (from payment system)
Orders       → JSON                        Refunds   → Azure SQL Database (JDBC)
Addresses    → TSV (tab-separated)
Memberships  → PNG images (binary)
```

### How Data Arrives in Production:
> Files are copied to ADLS by tools like **Azure Data Factory, Azure Event Hub**, or other pipeline tools — we don't build that part. Our job starts once data lands in the storage layer.

**⚠️ Production Tip:** In real projects, data engineers own from the landing zone onwards. Understanding how data arrives (ADF pipelines, streaming, batch drops) helps you debug when files are missing or late.

---

## 3. 🏗️ Environment Setup — ADLS & Unity Catalog

### ADLS Folder Structure (Course — Azure):
```
deacourseextdl  (storage account)
  └── gizmobox  (container)
        └── landing/
              ├── operational-data/
              │     ├── customers/
              │     ├── addresses/
              │     ├── orders/
              │     └── memberships/
              └── external-data/
                    └── payments/
```

### Your Setup (Community Edition — Volumes):
```
gizmobox  (catalog)
  └── landing  (schema)
        ├── operational/  (volume)
        │     ├── customers/
        │     ├── addresses/
        │     ├── orders/
        │     ├── memberships/
        │     ├── customers_autoloader/
        │     └── customers_stream/
        └── external/  (volume)
              └── payments/
```

### Unity Catalog Setup — Key SQL Commands:

```sql
-- See all existing catalogs
SHOW CATALOGS;

-- Create a new catalog
CREATE CATALOG IF NOT EXISTS gizmobox
MANAGED LOCATION 'abfss://gizmobox@deacourseextdl.dfs.core.windows.net/'
COMMENT 'GizmoBox Ecommerce Data Lakehouse';

-- Create schemas inside catalog
CREATE SCHEMA IF NOT EXISTS gizmobox.landing;
CREATE SCHEMA IF NOT EXISTS gizmobox.bronze;
CREATE SCHEMA IF NOT EXISTS gizmobox.silver;
CREATE SCHEMA IF NOT EXISTS gizmobox.gold;

-- Set active catalog (so you don't need to prefix everything)
USE CATALOG gizmobox;
```

**Your version (Community Edition — no MANAGED LOCATION needed):**
```sql
CREATE CATALOG IF NOT EXISTS gizmobox
COMMENT 'GizmoBox Ecommerce Data Lakehouse';
```

---

## 4. 🔍 Querying Files Directly with SQL

> One of Spark's most powerful features — query files directly **without creating a table first**

### Syntax:
```sql
SELECT * FROM <file_format>.`<file_path>`
--              ↑               ↑
--         format name      backticks (not quotes!)
```

> ⚠️ **Important:** Always use **backticks** `` ` `` around the file path, NOT single or double quotes.

### Examples:

```sql
-- Single file
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/customers_2024_10.json`

-- Multiple specific files
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/customers_2024_10.json`,
              json.`/Volumes/gizmobox/landing/operational/customers/customers_2024_11.json`

-- Entire folder (all files at once) ← most common in production
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`

-- CSV/TSV
SELECT * FROM csv.`/Volumes/gizmobox/landing/operational/addresses/`

-- Parquet
SELECT * FROM parquet.`/path/to/folder/`
```

**⚠️ Production Tip:** Always query the **folder** not individual files in production. New files get dropped into the folder daily/monthly — querying the folder picks them all up automatically without changing your SQL.

---

## 5. 👁️ Views — 3 Types

Views let you save a query as a named object so others can query it without knowing the underlying file path. Critical for the Bronze layer.

### Syntax to Create a View on Raw Files:
```sql
CREATE OR REPLACE VIEW catalog.schema.view_name
AS
SELECT * FROM json.`/path/to/files/`
```

### All 3 View Types Compared:

| Type | Syntax | Scope | Persists after? |
|---|---|---|---|
| **Permanent View** | `CREATE OR REPLACE VIEW` | Catalog/Schema | ✅ Always |
| **Temporary View** | `CREATE OR REPLACE TEMPORARY VIEW` | Current session only | ❌ Gone when session ends |
| **Global Temporary View** | `CREATE OR REPLACE GLOBAL TEMP VIEW` | All sessions in same Spark app | ❌ Gone when Spark app ends |

---

### 1. Permanent View (Bronze Layer View):
```sql
-- Stored in catalog.schema — available to everyone always
CREATE OR REPLACE VIEW gizmobox.bronze.v_customers
AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`
```

Access it like a regular table:
```sql
SELECT * FROM gizmobox.bronze.v_customers;
```

---

### 2. Temporary View (Session-scoped):
```sql
-- Only available in this notebook session — gone when you detach cluster
CREATE OR REPLACE TEMPORARY VIEW tv_customers
AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`
```

```sql
SELECT * FROM tv_customers;  -- no catalog.schema prefix needed
```

---

### 3. Global Temporary View (App-scoped):
```sql
-- Available across notebooks in same Spark application
-- Stored in special 'global_temp' schema
CREATE OR REPLACE GLOBAL TEMP VIEW gtv_customers
AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`
```

```sql
-- Must use global_temp prefix to query it
SELECT * FROM global_temp.gtv_customers;
```

---

### When to Use Which?

| Use Case | View Type |
|---|---|
| Bronze layer — shared with team, always available | **Permanent View** |
| Quick ad-hoc exploration in your notebook | **Temporary View** |
| Share across multiple notebooks in same session | **Global Temp View** |
| Production pipelines | **Permanent View** or **Delta Table** |

**⚠️ Production Tip:** In production Bronze layer, prefer **Delta Tables** over views for raw data. Views re-read files every time they're queried — Delta tables are faster and support ACID transactions, time travel, and schema enforcement. Use views for abstraction on top of Delta tables in Silver/Gold.

---

## 📌 Day 4 — Quick Recap

```
ETL Scope        → Query files → Validate → Transform → Load to Delta tables
GizmoBox Sources → Customers/Orders (JSON), Addresses (TSV), 
                   Memberships (PNG), Payments (CSV), Refunds (Azure SQL JDBC)

Direct File Query Syntax:
  SELECT * FROM json.`/path/to/folder/`   ← backticks, folder not file

3 View Types:
  Permanent View     → always available, stored in catalog.schema
  Temporary View     → current session only
  Global Temp View   → all sessions until Spark app ends, use global_temp.name

Unity Catalog SQL:
  SHOW CATALOGS
  CREATE CATALOG IF NOT EXISTS name
  USE CATALOG name
```

---

*Day 4 ✅ — Notes ongoing, more to be added as session continues*
*Next: Extracting & querying complex JSON, JDBC external tables, Bronze layer tables*
